# Comprehensive Exploratory Data Analysis: Renewal Calls Dataset
## ds_p.default.renewal_calls

This analysis explores customer renewal call data to understand patterns in membership renewal decisions, churn factors, competitor mentions, pricing discussions, and customer satisfaction.

In [0]:
# Load the renewal calls data
df = spark.table("ds_p.default.renewal_calls")

# Display basic information
print(f"Total number of records: {df.count():,}")
print(f"Number of columns: {len(df.columns)}")
print(f"\nColumn names and types:")
df.printSchema()

# Show first few rows
print("\nFirst 5 rows:")
display(df.limit(5))

In [0]:
# Get record count and column statistics
total_records = df.count()
num_columns = len(df.columns)

print(f"Dataset Shape:")
print(f"  Total Records: {total_records:,}")
print(f"  Total Columns: {num_columns}")
print(f"\n" + "="*60)

# Get numeric column statistics
from pyspark.sql.functions import col

numeric_cols = [field.name for field in df.schema.fields if field.dataType.typeName() in ['long', 'int', 'double', 'float']]
if numeric_cols:
    print(f"\nNumeric Column Statistics:")
    display(df.select(numeric_cols).summary())

In [0]:
from pyspark.sql.functions import min, max, countDistinct, year, month, dayofweek, date_format

# Analyze date ranges
date_stats = df.select(
    min("Call_Date").alias("Earliest_Call"),
    max("Call_Date").alias("Latest_Call"),
    countDistinct("Call_Date").alias("Unique_Days"),
    countDistinct("Call_Year").alias("Unique_Years")
).collect()[0]

print("Date Range Coverage:")
print(f"  Earliest Call: {date_stats['Earliest_Call']}")
print(f"  Latest Call: {date_stats['Latest_Call']}")
print(f"  Unique Days: {date_stats['Unique_Days']:,}")
print(f"  Unique Years: {date_stats['Unique_Years']}")
print(f"\n" + "="*60)

# Calls by year
print("\nCalls by Year:")
calls_by_year = df.groupBy("Call_Year").count().orderBy("Call_Year")
display(calls_by_year)

# Calls by month
print("\nCalls by Month:")
calls_by_month = df.withColumn("Month", date_format("Call_Date", "yyyy-MM")) \
    .groupBy("Month").count() \
    .orderBy("Month")
display(calls_by_month)

In [0]:
from pyspark.sql.functions import col, count, when, isnan, isnull

# Calculate missing values for each column
missing_data = []

for column in df.columns:
    null_count = df.filter(col(column).isNull()).count()
    null_percentage = (null_count / total_records) * 100
    missing_data.append((column, null_count, null_percentage))

# Create DataFrame and sort by null percentage
missing_df = spark.createDataFrame(missing_data, ["Column_Name", "Null_Count", "Null_Percentage"]) \
    .orderBy(col("Null_Percentage").desc())

print("Missing Values Analysis:")
print(f"Columns with missing data:\n")
display(missing_df.filter(col("Null_Count") > 0))

In [0]:
from pyspark.sql.functions import col, count, round as spark_round

# Membership Renewal Decision distribution
print("Membership Renewal Decision Distribution:")
renewal_dist = df.groupBy("Membership_Renewal_Decision") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(renewal_dist)

# Desire to Cancel distribution
print("\nDesire to Cancel Distribution:")
cancel_dist = df.groupBy("Desire_To_Cancel") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(cancel_dist)

# Churn Category analysis
print("\nChurn Category Distribution:")
churn_dist = df.filter(col("Churn_Category").isNotNull()) \
    .groupBy("Churn_Category") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(churn_dist)

In [0]:
# Customer Reaction Categories
print("Customer Reaction Categories:")
reaction_dist = df.filter(col("Customer_Reaction_Category").isNotNull()) \
    .groupBy("Customer_Reaction_Category") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(reaction_dist)

# Agent Renewal Pitch Categories
print("\nAgent Renewal Pitch Categories:")
pitch_dist = df.filter(col("Agent_Renewal_Pitch_Category").isNotNull()) \
    .groupBy("Agent_Renewal_Pitch_Category") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(pitch_dist)

# Customer Renewal Response Categories
print("\nCustomer Renewal Response Categories:")
response_dist = df.filter(col("Customer_Renewal_Response_Category").isNotNull()) \
    .groupBy("Customer_Renewal_Response_Category") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(response_dist)

In [0]:
# Serious Complaints
print("Serious Complaint Indicator:")
serious_complaint = df.groupBy("Serious_Complaint") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(serious_complaint)

# Other Complaints
print("\nOther Complaint Indicator:")
other_complaint = df.groupBy("Other_Complaint") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(other_complaint)

# Complaint Categories
print("\nComplaint Categories:")
complaint_categories = df.filter(col("Complaint_Category").isNotNull()) \
    .groupBy("Complaint_Category") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(complaint_categories)

In [0]:
# Explicit Competitor Mention
print("Explicit Competitor Mention:")
explicit_competitor = df.groupBy("Explicit_Competitor_Mention") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(explicit_competitor)

# Implied Competitor Mention
print("\nImplied Competitor Mention:")
implied_competitor = df.groupBy("Implied_Competitor_Mention") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(implied_competitor)

# Switching Intent
print("\nExplicit Switching Intent:")
explicit_switch = df.groupBy("Explicit_Switching_Intent") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(explicit_switch)

print("\nImplied Switching Intent:")
implied_switch = df.groupBy("Implied_Switching_Intent") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(implied_switch)

# Mentioned Competitors
print("\nMentioned Competitors:")
mentioned_competitors = df.filter(col("Mentioned_Competitors").isNotNull()) \
    .groupBy("Mentioned_Competitors") \
    .agg(count("*").alias("Count")) \
    .orderBy(col("Count").desc()) \
    .limit(20)
display(mentioned_competitors)

In [0]:
# Discussion on Price Increase
print("Discussion on Price Increase:")
price_discussion = df.groupBy("Discussion_on_Price_Increase") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(price_discussion)

# Renewal Impact Due to Price Increase
print("\nRenewal Impact Due to Price Increase:")
renewal_impact = df.filter(col("Renewal_Impact_Due_to_Price_Increase").isNotNull()) \
    .groupBy("Renewal_Impact_Due_to_Price_Increase") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(renewal_impact)

# Discount or Waiver Requested
print("\nDiscount or Waiver Requested:")
discount_requested = df.groupBy("Discount_or_Waiver_Requested") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(discount_requested)

# Discount Offered
print("\nDiscount Offered:")
discount_offered = df.groupBy("Discount_Offered") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(discount_offered)

In [0]:
# Call Direction
print("Call Direction:")
call_direction = df.groupBy("Call_Direction") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(call_direction)

# Analysed Call status
print("\nAnalysed Call Status:")
analysed_status = df.groupBy("Analysed_Call") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(analysed_status)

# Agent Renewal Initiation
print("\nAgent Renewal Initiation:")
agent_initiation = df.groupBy("Agent_Renewal_Initiation") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(agent_initiation)

In [0]:
# Renewal Decision vs Price Discussion
print("Renewal Decision vs Price Discussion:")
renewal_vs_price = df.filter(col("Membership_Renewal_Decision").isNotNull()) \
    .groupBy("Membership_Renewal_Decision", "Discussion_on_Price_Increase") \
    .agg(count("*").alias("Count")) \
    .orderBy("Membership_Renewal_Decision", col("Count").desc())
display(renewal_vs_price)

# Renewal Decision vs Competitor Mention
print("\nRenewal Decision vs Explicit Competitor Mention:")
renewal_vs_competitor = df.filter(col("Membership_Renewal_Decision").isNotNull()) \
    .groupBy("Membership_Renewal_Decision", "Explicit_Competitor_Mention") \
    .agg(count("*").alias("Count")) \
    .orderBy("Membership_Renewal_Decision", col("Count").desc())
display(renewal_vs_competitor)

# Renewal Decision vs Serious Complaint
print("\nRenewal Decision vs Serious Complaint:")
renewal_vs_complaint = df.filter(col("Membership_Renewal_Decision").isNotNull()) \
    .groupBy("Membership_Renewal_Decision", "Serious_Complaint") \
    .agg(count("*").alias("Count")) \
    .orderBy("Membership_Renewal_Decision", col("Count").desc())
display(renewal_vs_complaint)

In [0]:
# Reason for Renewal Categories
print("Reason for Renewal Categories:")
renewal_reasons = df.filter(col("Reason_For_Renewal_Category").isNotNull()) \
    .groupBy("Reason_For_Renewal_Category") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(renewal_reasons)

# Justification Categories (when provided)
print("\nJustification Categories:")
justification = df.filter(col("Justification_Category").isNotNull()) \
    .groupBy("Justification_Category") \
    .agg(count("*").alias("Count")) \
    .withColumn("Percentage", spark_round((col("Count") / total_records) * 100, 2)) \
    .orderBy(col("Count").desc())
display(justification)

## Key Insights from Renewal Calls Data

### Data Overview
* The dataset contains customer renewal call records from 2025
* Comprehensive tracking of 37 different features covering renewal decisions, customer reactions, agent interactions, pricing discussions, and competitor analysis

### Critical Findings (will be populated after running analysis):

**Renewal Patterns:**
* Membership renewal decision distribution shows the percentage of renewals vs non-renewals
* Desire to cancel indicates early warning signals

**Customer Satisfaction:**
* Complaint categories reveal main pain points
* Customer reaction patterns indicate sentiment during calls

**Competitive Landscape:**
* Competitor mentions and switching intent tracking
* Specific competitors being mentioned by customers

**Pricing Impact:**
* Price increase discussions and their impact on renewal decisions
* Discount requests and offers analysis

**Agent Performance:**
* Agent renewal pitch effectiveness
* Response strategies to cancellation requests

**Recommendations:**
1. Focus on addressing top complaint categories
2. Develop counter-strategies for main competitor threats
3. Optimize pricing communication and discount strategies
4. Enhance agent training based on successful renewal patterns
5. Monitor early warning signals (desire to cancel, serious complaints)